# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1: Hand-written rules fail when signals are tangled.**
- **Label Source**: This is based on comparing rule-based flags against observed future performance (like rankings dropping).
- **Validation Check**: If the validation compared the rule to an ML model on a held-out test set, the claim holds. The validation design properly isolates the complexity of tangled signals.

**Finding 2: Word count and engagement rate are strong predictors of search position.**
- **Label Source**: Observed `avg_position` from trailing 90-day data.
- **Validation Check**: A model trained to predict position might rely on word count, but without a time-aware split, we cannot claim causal impact. We can only claim an *observed directional correlation*. The validation supports finding a relationship, but not predicting Google's algorithm.

In [1]:
# No code needed for section 1


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
df = df[df['avg_position'] > 0].copy()
df['word_count'] = df['word_count'].fillna(0)
df['search_volume'] = df['search_volume'].fillna(df['search_volume'].median())

features = ['search_volume', 'competition', 'word_count', 'content_age_days']
target = 'avg_position'

# Leaky Random Split
X_train_rand, X_test_rand, y_train_rand, y_test_rand = train_test_split(df[features], df[target], test_size=0.2, random_state=42)
rf_rand = RandomForestRegressor(n_estimators=50, max_depth=6, random_state=42, n_jobs=-1)
rf_rand.fit(X_train_rand, y_train_rand)
rand_mae = mean_absolute_error(y_test_rand, rf_rand.predict(X_test_rand))

# Honest Grouped Split (by client_id)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df[features], df[target], groups=df['client_id']))
X_train_grp, y_train_grp = df.iloc[train_idx][features], df.iloc[train_idx][target]
X_test_grp, y_test_grp = df.iloc[test_idx][features], df.iloc[test_idx][target]

rf_grp = RandomForestRegressor(n_estimators=50, max_depth=6, random_state=42, n_jobs=-1)
rf_grp.fit(X_train_grp, y_train_grp)
grp_mae = mean_absolute_error(y_test_grp, rf_grp.predict(X_test_grp))

print(f"Leaky Random Split MAE: {rand_mae:.2f}")
print(f"Honest Grouped Split MAE: {grp_mae:.2f}")
print("Notice that the honest split performs worse (higher MAE) because the model cannot memorize client-specific patterns.")

Leaky Random Split MAE: 10.02
Honest Grouped Split MAE: 11.51
Notice that the honest split performs worse (higher MAE) because the model cannot memorize client-specific patterns.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

- **Label-derived features:** None of our features (`search_volume`, `word_count`, `competition`, `content_age_days`) are derived from `avg_position`.
- **Future windows:** All metrics are from the trailing 90 days. We are not using `impressions_last_30d` to predict `avg_position` of the last 30 days.
- **Product flags:** We did not include any internal FlyRank scores or "needs_refresh" flags.

In [3]:
print("Final feature set audited:", features)
print("Target variable:", target)

Final feature set audited: ['search_volume', 'competition', 'word_count', 'content_age_days']
Target variable: avg_position


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original:** "Our Random Forest model proves that increasing word count directly boosts a page's search ranking by 5 positions."

**Rewritten (Safe):** "Our signal analysis measured a directional relationship between higher word counts and better average search positions within the observed dataset, providing decision-support for content expansion strategies."

In [4]:
# Just printing the claims to keep the cell output clean
print("Claim Rewrite Complete.")

Claim Rewrite Complete.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.